[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jibby2k1/SPS_Curriculum/blob/main/Intro_Time_Series/Beyond_Kalman.ipynb)


**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Beyond Kalman: EKF, UKF & Particle Filters

The [Kalman filter](./Intro_AdFilt_KF.ipynb) is optimal for linear dynamics and Gaussian noise. Reality is neither. Three sessions on the escalation ladder: linearize (EKF), sample deterministically (UKF), sample massively (particle filter) — each demonstrated on problems where its predecessor fails.

## 1. Pre-requisites

[Adaptive Filtering: Kalman](./Intro_AdFilt_KF.ipynb) — this workshop assumes its notation and predict/update instincts.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(2)

---
### 🕐 Session 1 of 3 — *The Extended Kalman Filter* (~35 min)
**Goal:** linearize the model at the current estimate; track a pendulum.
**Builds on:** [Kalman workshop](./Intro_AdFilt_KF.ipynb). &nbsp; **Feeds into:** Session 2 (UKF).

---

## 2. EKF: Pretend It's Linear (Locally)

💡 **Intuition.** Nonlinear dynamics $f(\mathbf{x})$ break the Kalman derivation. EKF's fix is a first-order Taylor patch: propagate the *mean* through the true $f$, but propagate the *covariance* through $f$'s Jacobian at the current estimate — a fresh linearization every step. It works beautifully while the uncertainty stays small enough that $f$ is locally straight; it lies when curvature bites within one standard deviation.

In [ ]:
# Pendulum: x = [angle, angular velocity], observe only sin(angle) (e.g. a horizontal position sensor)

# YOUR CODE HERE


**What just happened.** A steady-state angle RMSE of **0.0082 rad** — about half a degree — from a filter that started at `[1.5, 0.5]` when the truth was `[2.2, 0]`, and that never observes the angle directly. Compare against the faint `arcsin(z)` scatter: that is the same measurement stream decoded without any dynamics, and it is both far noisier and outright wrong wherever $|\theta| > \pi/2$, because $\arcsin$ cannot tell $\theta$ from $\pi - \theta$.

That contrast is the argument for state estimation in one plot. A single measurement of $\sin\theta$ is genuinely ambiguous, and no amount of cleverness resolves it instant by instant. The filter succeeds because it fuses the measurement with a *model* — it knows the pendulum has momentum and roughly where it was heading, so the impossible branch is inconsistent with everything seen so far. Information from the past is doing at least as much work here as the current sample.

**Where the approximation actually sits.** The mean went through the true nonlinear $f$; only the covariance went through the Jacobian, and only that step is approximate. This is why EKF's point estimate can look excellent while its reported uncertainty is subtly wrong — and the reported uncertainty is what drives the gain. The failure mode is therefore quiet: nothing throws, the plot looks fine, and the filter is overconfident.

**And this problem is friendlier than it appears.** The angle uncertainty stays small once the filter has locked on, so within any one step $\sin$ is nearly straight across the width of the belief, and the tangent-line patch is a good one. Widen the belief — a worse initial guess, a larger $Q$, or sparser measurements — and curvature starts to bite inside a single standard deviation, at which point the Jacobian stops describing what the distribution actually does. Session 2 builds exactly that case and shows the tangent line failing in a way no amount of tuning fixes.

---
### 🕐 Session 2 of 3 — *The Unscented Kalman Filter* (~35 min)
**Goal:** replace Jacobians with sigma points; win when curvature bites.
**Builds on:** Session 1. &nbsp; **Feeds into:** Session 3 (particle filters).

---

## 3. UKF: Sample the Belief, Not the Slope

💡 **Intuition.** EKF pushes *one* point and a slope through $f$. The unscented transform pushes a handful of **sigma points** — deterministically placed at the mean ± scaled covariance directions — through the *true* nonlinear $f$, then refits mean and covariance to where they landed. No Jacobians (great when $f$ is ugly or black-box), and accurate to second order instead of first. Motto: *it's easier to approximate a distribution than a nonlinear function.*

In [ ]:
# The classic curvature demo: push a Gaussian through polar→Cartesian
# Monte Carlo truth
# EKF-style: linearize at the mean
# Unscented transform

# YOUR CODE HERE


**What just happened.** Three estimates of the same mean: Monte Carlo over 4000 samples gives $[-0.005, 0.940]$, the unscented transform gives $[0.000, 0.941]$ from **five** points, and the EKF-style linearization insists on $[0.000, 1.000]$.

And we can do better than comparing against Monte Carlo, because this problem has an exact answer. With $\theta \sim \mathcal{N}(\pi/2, \sigma^2)$ and $\sigma = 0.35$, the expected $y$-coordinate is $E[r]\,E[\sin\theta] = \sin(\pi/2)\,e^{-\sigma^2/2} = e^{-0.06125} = \mathbf{0.9406}$. So the unscented transform's 0.941 is right to three decimals, the Monte Carlo 0.940 is right to the precision 4000 samples can offer, and the EKF's 1.000 is wrong by **5.9%** — a bias, not noise, and one that no amount of extra data would reduce.

**Why linearization is biased rather than merely inaccurate.** The tangent line at the mean is a straight map, and a straight map sends the centre of a distribution to the centre of its image. But the true map is curved: the cloud bends into an arc, and the mean of an arc lies *inside* the curve, not on it. This is Jensen's inequality wearing a geometric hat — $E[\sin\theta] < \sin(E[\theta])$ for a symmetric spread around a concave stretch. No first-order method can see this, because the effect is second-order in the spread, which is exactly what "EKF is accurate to first order, UKF to second" means in practice.

**The efficiency is the striking part.** Five deterministic points matched a 4000-sample Monte Carlo estimate — and beat it, since MC's $-0.005$ in the $x$-coordinate is sampling noise around a true value of exactly 0. Sigma points are a quadrature rule, chosen to reproduce the mean and covariance exactly and to capture the leading curvature correction, not a small random sample. Run this cell twice and the UT answer is identical while the MC answer moves.

**What this does not fix.** The unscented transform still summarises the outcome as one mean and one covariance — it fits a single Gaussian to the mapped points. Look at the scatter: the truth is a banana, and *no* Gaussian describes a banana well, whatever its mean. UKF gets the location right and still misrepresents the shape. When the belief is not merely curved but genuinely multi-hypothesis, one blob is not a bad approximation but the wrong data structure, which is where Session 3 begins.

---
### 🕐 Session 3 of 3 — *Particle Filters* (~40 min)
**Goal:** represent ANY belief with weighted samples; track through multimodality.
**Builds on:** Session 2.

---

## 4. When the Belief Isn't a Blob

💡 **Intuition.** EKF/UKF still summarize belief as mean + covariance — one Gaussian blob. Some problems are **multimodal**: a robot that observes only its *distance* to a wall genuinely can't distinguish left from right, and the honest belief is two blobs. The particle filter drops the Gaussian religion: carry $N$ weighted samples (*particles*), move each through the dynamics (with noise), reweight by measurement likelihood, and **resample** to cull the walking dead. It's the [LLN](../Intro_Math/Analysis/Independence.ipynb) as a filter — any shape of belief, at Monte Carlo prices.

In [ ]:
# 1-D corridor localization: robot observes |position| (symmetric!) plus noise

# YOUR CODE HERE


**What just happened.** Three snapshots of a belief that changes *shape*, not just position. At $t=2$ the particles form two clean clusters near $\pm 2$; at $t=20$ two are still there with one visibly thinning; by $t=55$ a single cluster sits on the truth. The filter began knowing nothing — particles spread uniformly over the whole corridor — and was never told the robot started on the left.

**The middle panel is the filter being right.** This is the part worth insisting on. Two modes is not confusion or a bug: the measurement is $|x|$, so positions $+2$ and $-2$ are *exactly* equally consistent with everything observed so far, and any filter reporting a single confident answer at $t=2$ would be wrong half the time it was run. The particle filter's advantage here is not accuracy, it is honesty about what the data does and does not determine.

**How the ambiguity breaks.** Both clusters drift rightward, because the motion model applies to every particle alike. But the true robot moves *away* from the centre while its mirror image moves *toward* it, so the two hypotheses predict $|x|$ evolving in opposite directions. Within a few steps the impostor's predicted measurements diverge from the real ones, its likelihoods collapse, and `rng.choice` stops selecting it. Motion resolved a symmetry that no single measurement could — the same lesson as the pendulum in Session 1, now with the ambiguity discrete rather than continuous.

**Watch the two lines doing the real work.** `weights = lik + 1e-300` prevents an all-zero weight vector from producing a division by zero when every particle is implausible — a genuine failure mode called particle depletion, not defensive padding. And `1/np.sum(weights**2) < Np/2` is the effective sample size test: resample only once the effective population has halved, because resampling costs diversity and injects its own noise. Resampling every step is a common beginner instinct and it measurably degrades the estimate.

**The bill.** Three thousand particles to localise in *one* dimension. Particle counts scale roughly exponentially with state dimension, so plain particle filtering is generally impractical past three to five dimensions without additional structure — Rao–Blackwellisation to handle the linear substructure analytically, or smarter proposal distributions. That is why the escalation rule ends where it does: reach for particles when the belief genuinely is not a blob, and not before. Here it was not a blob, and nothing simpler could have represented the question.

The disambiguation: both hypothesis clusters drift right, but only one keeps matching the measurements as the true robot crosses regions where $|x|$ evolves differently — the impostor cluster starves and dies at resampling. **No Gaussian filter can even represent the question.**

Costs to respect: $O(N_p)$ per step, particle death in high dimensions (the curse), and resampling noise. The escalation rule: EKF if mildly nonlinear, UKF if curvy or Jacobian-hostile, particles if multimodal or seriously non-Gaussian — never more machinery than the problem demands.

## 5. Conclusion

Linearize, sigma-sample, or particle-sample: three ways to keep the predict/update heartbeat when the world stops being linear. You now own the full state-estimation ladder from LMS to Monte Carlo.

---
## Where next

- [Recurrent Neural Networks](./Intro_RNN.ipynb) — *learn* the dynamics instead of modeling them.
- [Uncertainty in ML](../Intro_Mach_Learn/Uncertainty_in_ML.ipynb) — ensembles: particle filtering's spirit in deep learning.